[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/ch08_04_bert_transformer_from_scratch.ipynb)

# Chapter 8-4: BERT와 Transformer 밑바닥부터 구현하기

> **목표**: Transformer(트랜스포머) 아키텍처를 수식 수준에서 완전히 이해하고, PyTorch로 직접 구현한 뒤, BERT의 사전학습과 Fine-tuning(미세조정)까지 실습합니다.
>
> **대상**: 딥러닝을 처음 공부하는 분 (Attention(주의 메커니즘) 기초를 알면 좋습니다)
>
> **선수 지식**: ch08_03 (Attention) 노트북을 먼저 보시면 좋습니다.
>
> **소요 시간**: 약 4~6시간

---

## 전체 아키텍처 한눈에 보기

아래는 이 노트북에서 다룰 Transformer의 전체 구조를 한눈에 정리한 것입니다. 각 구성요소를 Part 1에서 이론으로 배우고, Part 2에서 직접 코드로 구현합니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│                    Transformer 전체 아키텍처                         │
│                                                                     │
│   ┌─── Encoder (인코더) ───┐    ┌─── Decoder (디코더) ───┐         │
│   │                        │    │                        │         │
│   │  입력 토큰              │    │  출력 토큰 (시프트)     │         │
│   │    ↓                   │    │    ↓                   │         │
│   │  Token Embedding       │    │  Token Embedding       │         │
│   │  + Positional Encoding │    │  + Positional Encoding │         │
│   │    ↓                   │    │    ↓                   │         │
│   │ ┌────────────────────┐ │    │ ┌────────────────────┐ │         │
│   │ │ Multi-Head         │ │    │ │ Masked Multi-Head  │ │         │
│   │ │ Self-Attention     │ │    │ │ Self-Attention     │ │         │
│   │ │ (양방향)           │ │    │ │ (단방향/미래 차단) │ │         │
│   │ └────────┬───────────┘ │    │ └────────┬───────────┘ │         │
│   │   Add & LayerNorm      │    │   Add & LayerNorm      │         │
│   │    ↓                   │    │    ↓                   │         │
│   │                        │ ──K,V──→ Cross-Attention   │         │
│   │                        │    │ │ (Q=Decoder, K,V=Enc)│         │
│   │                        │    │ └────────┬───────────┘ │         │
│   │                        │    │   Add & LayerNorm      │         │
│   │                        │    │    ↓                   │         │
│   │ ┌────────────────────┐ │    │ ┌────────────────────┐ │         │
│   │ │ Feed-Forward       │ │    │ │ Feed-Forward       │ │         │
│   │ │ Network (FFN)      │ │    │ │ Network (FFN)      │ │         │
│   │ └────────┬───────────┘ │    │ └────────┬───────────┘ │         │
│   │   Add & LayerNorm      │    │   Add & LayerNorm      │         │
│   │    ↓                   │    │    ↓                   │         │
│   │  × N층 반복            │    │  × N층 반복            │         │
│   └────────────────────────┘    └────────────────────────┘         │
│                                          ↓                         │
│                                   Linear + Softmax                 │
│                                          ↓                         │
│                                     출력 토큰                       │
└─────────────────────────────────────────────────────────────────────┘
```

> **비유**: Transformer는 "번역 사무소"와 같습니다. Encoder(인코더)는 원문을 깊이 이해하는 "분석팀"이고, Decoder(디코더)는 분석팀의 메모(K, V)를 참고하면서 한 글자씩 번역문을 작성하는 "번역팀"입니다. BERT는 이 중 분석팀(Encoder)만 사용하고, GPT는 번역팀(Decoder)만 사용합니다.

---

## Encoder vs Decoder 핵심 비교표

| 구분 | Encoder (인코더) | Decoder (디코더) |
|------|:---:|:---:|
| **Attention 방향** | 양방향 (모든 토큰 참조) | 단방향 (과거 토큰만 참조) |
| **Causal Mask** | 사용 안 함 | 사용 (미래 토큰 차단) |
| **Cross-Attention** | 없음 | 있음 (Encoder 출력 참조) |
| **대표 모델** | BERT, RoBERTa | GPT, LLaMA, Claude |
| **주요 용도** | 분류, 질의응답, NER | 텍스트 생성, 대화, 번역 |
| **비유** | 시험 문제를 전체 읽고 답하기 | 소설을 앞에서부터 한 줄씩 쓰기 |

---

## 목차

| 파트 | 내용 | 핵심 |
|------|------|------|
| **Part 1** | Transformer 이론 | 논문 동기, Scaled Dot-Product Attention(스케일 내적 주의) 수식 분해, Multi-Head(다중 헤드), Positional Encoding(위치 인코딩), Encoder/Decoder 구조 |
| **Part 2** | Transformer 밑바닥 구현 | PyTorch로 Attention, Multi-Head, Encoder Block 직접 구현 |
| **Part 3** | BERT 이해 & 활용 | MLM(마스크 언어 모델), NSP(다음 문장 예측), 입력 구조, DistilBERT Fine-tuning 감정 분류 실습 |
| **Part 4** | 정리 | Transformer 계보, 모델 비교, 핵심 수식 요약 |

---
## 환경 설정

In [ ]:
# === Colab 환경 설정 ===
# transformers: HuggingFace의 사전학습 모델 라이브러리 (BERT, GPT 등)
# datasets: HuggingFace의 데이터셋 라이브러리 (GLUE, SQuAD 등)
# accelerate: 분산 학습 및 mixed precision(혼합 정밀도) 지원
# 왜 설치하나? → Colab에는 기본 설치되어 있지 않으므로, 반드시 먼저 실행해야 합니다.
!pip install -q transformers datasets accelerate

In [ ]:
import torch  # PyTorch: 딥러닝 프레임워크
import torch.nn as nn  # 신경망 모듈 (Linear, LayerNorm 등)
import torch.nn.functional as F  # 활성화 함수, softmax 등
import numpy as np  # 수치 계산 라이브러리
import matplotlib.pyplot as plt  # 시각화 라이브러리
import math  # 수학 함수 (sqrt, sin, cos 등)
import warnings  # 경고 메시지 관리

# 불필요한 경고 숨기기 (학습에 집중하기 위해)
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Colab에서는 기본 폰트 사용)
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# 재현성을 위한 시드 고정 (같은 코드를 실행하면 항상 같은 결과가 나오도록)
torch.manual_seed(42)  # PyTorch 랜덤 시드
np.random.seed(42)  # NumPy 랜덤 시드

# GPU 사용 가능 여부 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # GPU가 있으면 GPU, 없으면 CPU
print(f'사용 디바이스: {device}')  # 현재 사용 중인 디바이스 출력

---
# Part 1: Transformer 이론

## 1.1 "Attention is All You Need" 논문의 핵심 동기

2017년 Google이 발표한 이 논문은 딥러닝 역사에서 가장 중요한 논문 중 하나입니다.

### 기존 RNN 기반 모델의 한계

RNN(Recurrent Neural Network, 순환 신경망)은 시퀀스(연속된 데이터)를 **순차적으로** 처리합니다:

```
h_1 → h_2 → h_3 → h_4 → ... → h_T
```

> **비유**: RNN은 "릴레이 달리기"와 같습니다. 선수(h_t)가 바통을 받아야만 다음 구간을 달릴 수 있습니다. 앞 선수가 느리면 뒷 선수도 기다려야 하죠.

이 구조의 문제점:

| 문제 | 설명 |
|------|------|
| **병렬화 불가** | $h_t$를 계산하려면 $h_{t-1}$이 필요 → GPU 병렬 처리 불가능 |
| **장거리 의존성** | 문장이 길어지면 앞쪽 정보가 희석됨 (Vanishing Gradient, 기울기 소실) |
| **학습 속도** | 순차 처리로 인해 학습이 매우 느림 |

### Transformer의 해결책

**"RNN을 완전히 제거하고, Attention(주의 메커니즘)만으로 시퀀스를 처리하자!"**

| 특성 | RNN | Transformer |
|------|-----|-------------|
| 처리 방식 | 순차적 ($O(T)$ 스텝) | **병렬** ($O(1)$ 스텝) |
| 장거리 의존성 | 간접적 (여러 스텝 거쳐야 함) | **직접적** (모든 위치 간 직접 연결) |
| 학습 속도 | 느림 | **빠름** (GPU 활용 극대화) |
| 최대 경로 길이 | $O(T)$ | $O(1)$ |

> **비유**: Transformer는 "단체 사진"과 같습니다. 모든 사람이 동시에 서로를 볼 수 있어서, 누가 누구 옆에 서야 할지 한 번에 결정할 수 있습니다.

핵심 아이디어: **Self-Attention(자기 주의 메커니즘)**으로 시퀀스 내 모든 위치 쌍의 관계를 **한 번에** 계산합니다.

## 1.2 Scaled Dot-Product Attention(스케일 내적 주의) 수식 완전 분해

Transformer의 핵심은 바로 이 수식입니다:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**쉽게 말하면**: "질문(Q)과 가장 관련 있는 정보(K)를 찾아서, 그 정보의 내용(V)을 가져오는 것"입니다. 도서관에서 검색어로 관련 책을 찾고, 그 책의 내용을 읽는 과정과 같습니다.

하나하나 분해해 보겠습니다.

---

### 1.2.1 Q, K, V 행렬의 의미

입력 시퀀스 $X \in \mathbb{R}^{T \times d_{model}}$이 있을 때 ($T$: 시퀀스 길이, $d_{model}$: 임베딩 차원):

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$

여기서 $W^Q, W^K \in \mathbb{R}^{d_{model} \times d_k}$, $W^V \in \mathbb{R}^{d_{model} \times d_v}$

| 행렬 | 이름 | 역할 | 비유 |
|------|------|------|------|
| $Q$ (Query, 질의) | 질의 | "나는 어떤 정보를 찾고 싶다" | 도서관에서 책을 찾는 **검색어** |
| $K$ (Key, 키) | 키 | "나는 이런 정보를 가지고 있다" | 책의 **제목/태그** |
| $V$ (Value, 값) | 값 | "실제로 전달할 정보" | 책의 **내용** |

**쉽게 말하면**: Q는 "내가 알고 싶은 것", K는 "각 단어가 제공하는 단서", V는 "실제 단어의 의미 정보"입니다.

**차원 정리**:
- $Q \in \mathbb{R}^{T \times d_k}$ (시퀀스 길이 $T$, 각 위치의 질의 벡터 $d_k$차원)
- $K \in \mathbb{R}^{T \times d_k}$ (시퀀스 길이 $T$, 각 위치의 키 벡터 $d_k$차원)
- $V \in \mathbb{R}^{T \times d_v}$ (시퀀스 길이 $T$, 각 위치의 값 벡터 $d_v$차원)

---

### 1.2.2 $QK^T$: 유사도 행렬 계산

$$QK^T \in \mathbb{R}^{T \times T}$$

이 행렬의 $(i, j)$ 원소는 **위치 $i$의 Query와 위치 $j$의 Key 사이의 내적(dot product)**입니다:

$$(QK^T)_{ij} = q_i \cdot k_j = \sum_{l=1}^{d_k} q_{il} \cdot k_{jl}$$

**쉽게 말하면**: 내적 값이 클수록 두 단어가 서로 관련이 높다는 뜻입니다. 이 값이 클수록 → 위치 $i$가 위치 $j$에 **더 많은 주의(attention)** 를 기울입니다.

예시 ("나는 학교에 간다" 문장):

```
         나는  학교에  간다
나는    [ 0.9   0.3   0.5 ]
학교에  [ 0.2   0.8   0.6 ]
간다    [ 0.4   0.7   0.9 ]
```

→ "간다"는 "학교에"에 높은 attention (0.7)을 줌 → 문맥적으로 자연스러움!

---

### 1.2.3 $\sqrt{d_k}$로 나누는 이유 (수학적 증명)

**문제**: $d_k$가 커지면 내적 값의 분산이 커져서 softmax가 극단적인 값을 만듭니다.

**쉽게 말하면**: 시험 점수의 편차가 너무 크면 1등만 상을 받고 나머지는 아무것도 못 받는 것처럼, softmax도 가장 큰 값에만 확률이 몰리는 문제가 생깁니다.

**증명**:

$q$와 $k$의 각 원소가 독립이고 평균 0, 분산 1인 확률변수라고 가정합니다.

내적 $q \cdot k = \sum_{l=1}^{d_k} q_l \cdot k_l$의 기대값과 분산을 구하면:

$$E[q \cdot k] = \sum_{l=1}^{d_k} E[q_l] \cdot E[k_l] = 0$$

$$\text{Var}(q \cdot k) = \sum_{l=1}^{d_k} \text{Var}(q_l \cdot k_l) = \sum_{l=1}^{d_k} E[q_l^2]E[k_l^2] = \sum_{l=1}^{d_k} 1 \cdot 1 = d_k$$

즉, **내적의 분산이 $d_k$에 비례**합니다!

- $d_k = 64$이면 → 내적의 표준편차가 $\sqrt{64} = 8$
- $d_k = 512$이면 → 내적의 표준편차가 $\sqrt{512} \approx 22.6$

softmax에 큰 값이 들어가면:
$$\text{softmax}([10, 0, 0]) \approx [1.0, 0.0, 0.0]$$

→ **거의 one-hot** 벡터가 되어 gradient(기울기)가 소실됩니다!

$\sqrt{d_k}$로 나누면:
$$\text{Var}\left(\frac{q \cdot k}{\sqrt{d_k}}\right) = \frac{d_k}{d_k} = 1$$

→ 분산이 항상 1로 유지 → softmax가 안정적으로 동작합니다.

---

### 1.2.4 softmax: 확률 분포로 변환

$$\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)_{ij} = \frac{\exp\left(\frac{q_i \cdot k_j}{\sqrt{d_k}}\right)}{\sum_{l=1}^{T} \exp\left(\frac{q_i \cdot k_l}{\sqrt{d_k}}\right)}$$

**쉽게 말하면**: 유사도 점수들을 0~1 사이의 확률로 변환합니다. 각 행의 확률 합은 반드시 1이 됩니다.

각 행이 **확률 분포** (합이 1)가 됩니다:
- 위치 $i$가 다른 모든 위치에 대해 **주의를 얼마나 분배할지** 결정

---

### 1.2.5 최종 출력: Attention 가중합

$$\text{Output} = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Attention 가중치 행렬 $A \in \mathbb{R}^{T \times T}$와 $V \in \mathbb{R}^{T \times d_v}$를 곱하면:

$$\text{Output}_i = \sum_{j=1}^{T} A_{ij} \cdot v_j$$

**쉽게 말하면**: 각 단어의 최종 표현은 문장 내 모든 단어의 Value를 "관련도에 따라 혼합"한 것입니다. 관련이 높은 단어의 정보를 많이, 관련이 낮은 단어의 정보를 적게 가져옵니다.

## 1.3 Multi-Head Attention(다중 헤드 주의 메커니즘)

### 왜 여러 Head(헤드)가 필요한가?

하나의 Attention Head는 하나의 "관점"으로만 관계를 학습합니다.

> **비유**: Multi-Head Attention은 **여러 명의 전문가가 같은 문서를 각자 다른 관점에서 읽는 것**과 같습니다. 문법 전문가는 주어-동사 관계를, 의미 전문가는 대명사가 무엇을 가리키는지를, 감정 전문가는 어조를 분석합니다.

예: "The cat sat on the mat because it was tired"
- Head 1: "it" → "cat" (대명사 참조 관계)
- Head 2: "sat" → "on the mat" (동사-전치사구 관계)
- Head 3: "tired" → "cat" (형용사-주어 관계)

**여러 Head가 서로 다른 관계 패턴을 동시에 학습**할 수 있습니다!

### 수식

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

각 head는:
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

**쉽게 말하면**: 각 Head가 독립적으로 Attention을 계산한 뒤, 결과를 이어붙이고(Concat) 하나의 출력으로 합칩니다.

여기서:
- $W_i^Q \in \mathbb{R}^{d_{model} \times d_k}$, $W_i^K \in \mathbb{R}^{d_{model} \times d_k}$, $W_i^V \in \mathbb{R}^{d_{model} \times d_v}$
- $W^O \in \mathbb{R}^{hd_v \times d_{model}}$ (출력 프로젝션)

### 차원 분할

논문에서는 Head 수로 차원을 균등 분할합니다:

$$d_k = d_v = \frac{d_{model}}{h}$$

예시 ($d_{model} = 512$, $h = 8$):
- 각 Head의 차원: $d_k = d_v = 512 / 8 = 64$
- 8개 Head의 출력을 이어붙이면: $8 \times 64 = 512 = d_{model}$ → 원래 차원 복원!

**핵심**: Multi-Head를 써도 총 연산량은 Single-Head와 거의 동일합니다.

```
Single Head:  d_model × d_model = 512 × 512 = 262,144 연산
Multi-Head:   h × (d_model × d_k) = 8 × (512 × 64) = 262,144 연산
```

## 1.4 Positional Encoding(위치 인코딩)

### 문제: Transformer에는 순서 정보가 없다!

RNN은 순차적으로 처리하므로 자연스럽게 위치 정보가 있지만, Transformer는 모든 위치를 **동시에** 처리합니다.

"나는 학교에 간다"와 "간다 학교에 나는"이 **같은 결과**를 낼 수 있습니다!

> **비유**: Transformer는 단어가 적힌 카드를 한꺼번에 받는 것과 같습니다. 카드에 순서 번호가 적혀있지 않으면, "나는 학교에 간다"와 "간다 나는 학교에"를 구별할 수 없습니다. 그래서 각 카드에 번호표(Positional Encoding)를 붙여줍니다.

### 해결: 위치 정보를 직접 더해준다

$$\text{Input} = \text{TokenEmbedding}(x) + \text{PositionalEncoding}(pos)$$

### 사인/코사인 Positional Encoding 공식

$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

여기서:
- $pos$: 토큰의 위치 (0, 1, 2, ...)
- $i$: 임베딩 차원의 인덱스 (0, 1, 2, ..., $d_{model}/2 - 1$)

**쉽게 말하면**: 각 위치마다 고유한 "지문"을 만들어 줍니다. 낮은 차원은 빠르게 변하는 사인파로 가까운 위치를 구분하고, 높은 차원은 느리게 변하는 사인파로 먼 위치 관계를 표현합니다.

### 직관적 이해

이 공식은 **이진수 카운터**와 비슷합니다:

```
pos=0:  0 0 0 0
pos=1:  0 0 0 1  ← 가장 낮은 비트가 빠르게 변함
pos=2:  0 0 1 0
pos=3:  0 0 1 1
pos=4:  0 1 0 0  ← 높은 비트는 느리게 변함
```

사인/코사인도 마찬가지:
- **낮은 차원 ($i$ 작음)**: 주기가 짧음 → 인접 위치를 구분
- **높은 차원 ($i$ 큼)**: 주기가 길음 → 먼 위치 관계를 인코딩

### 왜 사인/코사인인가?

1. **상대적 위치 표현 가능**: $PE(pos + k)$는 $PE(pos)$의 선형 변환으로 표현 가능
   $$\sin(\omega(pos+k)) = \sin(\omega \cdot pos)\cos(\omega k) + \cos(\omega \cdot pos)\sin(\omega k)$$
   → 모델이 상대적 거리를 쉽게 학습할 수 있음

2. **임의 길이 시퀀스에 대응**: 학습 데이터보다 긴 시퀀스에도 적용 가능

3. **값이 -1~1 범위**: 임베딩과 스케일이 비슷하여 더하기 적합

## 1.5 Transformer Encoder Block(인코더 블록)

Encoder Block은 다음 4단계로 구성됩니다:

```
입력 X
  │
  ├──→ Multi-Head Self-Attention ──→ + ──→ LayerNorm ──→ Z
  │         (Q=K=V=X)                │ (Residual)
  └──────────────────────────────────┘
  │
  ├──→ Feed-Forward Network ──→ + ──→ LayerNorm ──→ 출력
  │    (2층 MLP, ReLU)          │ (Residual)
  └────────────────────────────┘
```

> **비유**: Encoder Block은 "독해 과정"과 같습니다. Self-Attention은 "문장을 읽으며 어떤 단어가 어떤 단어와 관련 있는지 파악하는 단계"이고, FFN은 "파악한 관계를 바탕으로 각 단어의 의미를 깊이 처리하는 단계"입니다. Residual Connection은 "원래 이해하고 있던 것을 잊지 않도록 메모를 유지하는 것"입니다.

### 1.5.1 Residual Connection(잔차 연결)

$$\text{Output} = \text{LayerNorm}(X + \text{SubLayer}(X))$$

**쉽게 말하면**: 입력 X를 그대로 출력에 더해줍니다. SubLayer(서브레이어)는 "변화량(잔차)"만 학습하면 되므로 학습이 훨씬 쉬워집니다.

**왜 필요한가?**
- 깊은 네트워크에서 gradient(기울기)가 잘 흐르도록 **지름길**을 제공
- $X$를 그대로 더해주므로, SubLayer는 **변화량(잔차)** 만 학습하면 됨
- ResNet(2015)에서 영감을 받은 기법

### 1.5.2 Layer Normalization(층 정규화) vs Batch Normalization(배치 정규화)

| 특성 | Batch Normalization | Layer Normalization |
|------|--------------------|--------------------|
| 정규화 축 | 배치 차원 (같은 특성의 여러 샘플) | 특성 차원 (한 샘플의 모든 특성) |
| 수식 | $\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$ | $\hat{x} = \frac{x - \mu_L}{\sqrt{\sigma_L^2 + \epsilon}}$ |
| 배치 크기 의존 | O (작은 배치에서 불안정) | X (배치 크기 무관) |
| 시퀀스 길이 | 가변 길이 처리 어려움 | **가변 길이 자연스럽게 처리** |
| Transformer 적합성 | 부적합 | **적합** |

Layer Normalization 수식:
$$\mu_L = \frac{1}{d}\sum_{i=1}^{d} x_i, \quad \sigma_L^2 = \frac{1}{d}\sum_{i=1}^{d}(x_i - \mu_L)^2$$
$$\text{LayerNorm}(x) = \gamma \odot \frac{x - \mu_L}{\sqrt{\sigma_L^2 + \epsilon}} + \beta$$

여기서 $\gamma, \beta$는 학습 가능한 파라미터 (스케일, 시프트)

**쉽게 말하면**: 각 샘플 내에서 값들의 평균을 0, 분산을 1로 맞춰주는 것입니다. 이렇게 하면 학습이 안정적으로 진행됩니다.

### 1.5.3 Feed-Forward Network(순방향 신경망, FFN)

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

- $W_1 \in \mathbb{R}^{d_{model} \times d_{ff}}$: 차원을 확장 (보통 $d_{ff} = 4 \times d_{model}$)
- $W_2 \in \mathbb{R}^{d_{ff} \times d_{model}}$: 차원을 축소하여 원래로 복원
- 각 위치에 **독립적으로** 적용 (position-wise)

**쉽게 말하면**: Attention이 "어디를 볼지" 결정했다면, FFN은 "본 것을 어떻게 처리할지" 결정합니다. 차원을 4배로 늘렸다가 다시 줄이는 "병목 구조"로, 비선형적인 패턴을 학습합니다.

## 1.6 Transformer Decoder Block(디코더 블록)

Decoder Block은 Encoder보다 한 층이 더 있습니다:

```
타겟 입력 Y
  │
  ├──→ Masked Multi-Head Self-Attention ──→ + ──→ LayerNorm ──→ Z₁
  │         (Q=K=V=Y, 미래 마스킹)           │
  └──────────────────────────────────────────┘
  │
  ├──→ Multi-Head Cross-Attention ──→ + ──→ LayerNorm ──→ Z₂
  │    (Q=Z₁, K=V=Encoder출력)        │
  └──────────────────────────────────┘
  │
  ├──→ Feed-Forward Network ──→ + ──→ LayerNorm ──→ 출력
  │                             │
  └────────────────────────────┘
```

### 1.6.1 Causal Mask(인과적 마스크)

번역 등 생성 과제에서는 **미래 토큰을 볼 수 없어야** 합니다.

> **비유**: 시험에서 아직 나오지 않은 다음 문제의 답을 미리 보면 안 되는 것처럼, Decoder도 아직 생성하지 않은 미래 단어를 참고하면 안 됩니다.

"I love you"를 "나는 너를 사랑해"로 번역할 때:
- "나는"을 생성할 때 → "너를", "사랑해"를 참조하면 안 됨 (아직 생성 안 했으므로)

**Causal Mask 행렬** ($T=4$ 예시):

$$M = \begin{bmatrix} 0 & -\infty & -\infty & -\infty \\ 0 & 0 & -\infty & -\infty \\ 0 & 0 & 0 & -\infty \\ 0 & 0 & 0 & 0 \end{bmatrix}$$

이 마스크를 Attention score에 더합니다:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$

$-\infty$를 더하면 softmax 후 0이 됩니다:
$$\text{softmax}(-\infty) = \frac{e^{-\infty}}{\sum} = \frac{0}{\sum} = 0$$

→ 미래 위치의 Attention 가중치가 0이 되어 **미래 정보 차단**!

### 1.6.2 Cross-Attention(교차 주의 메커니즘)

Decoder의 2번째 Attention은 **Cross-Attention** (교차 주의)입니다:
- **Query**: Decoder의 현재 상태 ("지금 무슨 정보가 필요한가?")
- **Key, Value**: Encoder의 출력 ("원문에서 관련 정보 검색")

**쉽게 말하면**: 번역할 때 원문(Encoder 출력)을 다시 참고하면서 번역문을 만드는 것입니다. "이 부분을 번역하려면 원문의 어디를 봐야 하지?"라고 질문하는 과정입니다.

→ Encoder-Decoder 사이의 다리 역할!

## 1.7 Encoder vs Decoder 완전 비교 (BERT vs GPT)

| 특성 | Encoder | Decoder |
|------|---------|--------|
| Self-Attention | 양방향 (모든 위치 참조) | **단방향** (과거 위치만 참조) |
| Mask | 없음 (패딩 마스크만) | Causal Mask(인과적 마스크) 사용 |
| Cross-Attention | 없음 | Encoder 출력과 Cross-Attention |
| 대표 모델 | **BERT** (Encoder-only) | **GPT** (Decoder-only) |
| 적합한 과제 | 분류, NER(개체명 인식), 질의응답 | **텍스트 생성**, 번역 |
| 입력 처리 | 전체 문장을 한 번에 봄 | 왼쪽→오른쪽으로 순차 생성 |

### 전체 Transformer 아키텍처 요약

```
원본 논문의 Transformer (Encoder-Decoder):

  [입력 문장]              [타겟 문장]
      │                        │
  Embedding                Embedding
  + PosEnc                 + PosEnc
      │                        │
  ┌────────┐              ┌────────┐
  │Encoder │──────────────│Decoder │
  │Block   │   (K, V)     │Block   │
  │× N층   │              │× N층   │
  └────────┘              └────────┘
                               │
                          Linear + Softmax
                               │
                          [출력 토큰]
```

논문 기본 설정:
- $N = 6$ (Encoder/Decoder 각 6층)
- $d_{model} = 512$
- $h = 8$ (8개 Attention Head)
- $d_k = d_v = 64$
- $d_{ff} = 2048$

---
# Part 2: Transformer 밑바닥 구현 (PyTorch)

이제 Part 1에서 배운 수식을 한 줄 한 줄 코드로 옮기겠습니다.

> **구현 순서**: Scaled Dot-Product Attention → Multi-Head Attention → Positional Encoding → Encoder Block → 전체 Encoder → Causal Mask

## 2.1 Scaled Dot-Product Attention(스케일 내적 주의) 구현

In [ ]:
class ScaledDotProductAttention(nn.Module):
    """Scaled Dot-Product Attention(스케일 내적 주의) 구현.

    수식: Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V

    쉽게 말하면: Q(검색어)로 K(제목)을 검색하고,
    유사도가 높은 V(내용)를 가중합하여 결과를 만듭니다.

    Args:
        temperature: 스케일링 팩터 (보통 sqrt(d_k))
    """

    def __init__(self, temperature: float):
        """초기화 함수.

        Args:
            temperature: sqrt(d_k) 값. 내적을 이 값으로 나눠 분산을 조절합니다.
        """
        super().__init__()  # nn.Module 초기화
        self.temperature = temperature  # 스케일링 팩터 저장

    def forward(
        self,
        q: torch.Tensor,  # Query 행렬: (batch, seq_len, d_k)
        k: torch.Tensor,  # Key 행렬: (batch, seq_len, d_k)
        v: torch.Tensor,  # Value 행렬: (batch, seq_len, d_v)
        mask: torch.Tensor = None  # 마스크: (batch, 1, seq_len) 또는 (batch, seq_len, seq_len)
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Scaled Dot-Product Attention 순전파.

        Returns:
            output: Attention 가중합 결과 (batch, seq_len, d_v)
            attn_weights: Attention 가중치 행렬 (batch, seq_len, seq_len)
        """
        # 1단계: Q와 K의 내적 계산 → 유사도 점수
        # q: (batch, seq_len, d_k), k^T: (batch, d_k, seq_len)
        # 결과: (batch, seq_len, seq_len) → T×T 유사도 행렬
        # 왜 내적인가? → 두 벡터가 비슷한 방향이면 내적이 크므로 유사도를 측정하기에 적합
        attn_scores = torch.bmm(q, k.transpose(1, 2))  # 배치 행렬 곱 (batch matrix multiply)

        # 2단계: sqrt(d_k)로 스케일링 → 분산을 1로 조절
        # 왜 스케일링하나? → d_k가 크면 내적 값의 분산이 커져 softmax가 극단적으로 됨
        attn_scores = attn_scores / self.temperature

        # 3단계: 마스크 적용 (선택적)
        if mask is not None:
            # 왜 -1e9를 넣나? → softmax(-무한대) ≈ 0이므로 해당 위치의 가중치를 0으로 만듦
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)

        # 4단계: softmax로 확률 분포 변환
        # dim=-1: 마지막 차원(Key 방향)에 대해 softmax
        # 왜 softmax? → 유사도 점수를 합이 1인 확률 분포로 변환하여 가중치로 사용
        attn_weights = F.softmax(attn_scores, dim=-1)  # (batch, seq_len, seq_len)

        # 5단계: Attention 가중치와 Value의 가중합
        # attn_weights: (batch, seq_len, seq_len) × v: (batch, seq_len, d_v)
        # 결과: (batch, seq_len, d_v) → 각 위치의 문맥 벡터(context vector)
        # 왜 가중합? → 관련도 높은 단어의 정보를 많이, 낮은 단어는 적게 반영
        output = torch.bmm(attn_weights, v)

        return output, attn_weights

### Scaled Dot-Product Attention 동작 확인

In [ ]:
# === Scaled Dot-Product Attention 동작 확인 ===

# 하이퍼파라미터(초매개변수) 설정
batch_size = 1  # 배치 크기 (예제이므로 1)
seq_len = 4  # 시퀀스 길이 (4개의 토큰)
d_k = 8  # Key/Query 차원
d_v = 8  # Value 차원

# 랜덤 Q, K, V 생성 (실제로는 입력에 가중치 행렬을 곱해서 만듦)
q = torch.randn(batch_size, seq_len, d_k)  # Query: (1, 4, 8)
k = torch.randn(batch_size, seq_len, d_k)  # Key: (1, 4, 8)
v = torch.randn(batch_size, seq_len, d_v)  # Value: (1, 4, 8)

# Attention 모듈 생성
temperature = math.sqrt(d_k)  # sqrt(8) ≈ 2.83
attention = ScaledDotProductAttention(temperature)  # 스케일링 팩터 전달

# 순전파 실행
output, attn_weights = attention(q, k, v)  # Attention 계산

print(f'Q 형태: {q.shape}')  # (1, 4, 8)
print(f'K 형태: {k.shape}')  # (1, 4, 8)
print(f'V 형태: {v.shape}')  # (1, 4, 8)
print(f'출력 형태: {output.shape}')  # (1, 4, 8) → d_v와 같음
print(f'Attention 가중치 형태: {attn_weights.shape}')  # (1, 4, 4) → T×T 유사도 행렬
print(f'\nAttention 가중치 행렬 (각 행의 합 = 1.0):')  # softmax 결과이므로 합이 1
print(attn_weights[0].detach().numpy().round(3))  # 첫 번째 배치의 가중치
print(f'\n각 행의 합: {attn_weights[0].sum(dim=-1).detach().numpy()}')  # 확인: 모두 1.0

In [ ]:
# === Attention 가중치 히트맵 시각화 ===

# 예시 토큰 이름 (한국어 문장)
tokens = ['나는', '학교에', '매일', '간다']  # 4개의 토큰

# 히트맵 그리기
fig, ax = plt.subplots(figsize=(6, 5))  # 6×5 인치 크기의 그래프 생성
im = ax.imshow(  # 2D 행렬을 이미지로 표시
    attn_weights[0].detach().numpy(),  # (4, 4) Attention 가중치
    cmap='Blues',  # 파란색 계열 색상표
    vmin=0, vmax=1  # 값 범위: 0~1 (확률)
)

# 축 레이블 설정
ax.set_xticks(range(seq_len))  # x축 눈금 위치
ax.set_yticks(range(seq_len))  # y축 눈금 위치
ax.set_xticklabels(tokens)  # x축: Key (참조되는 쪽)
ax.set_yticklabels(tokens)  # y축: Query (주의를 기울이는 쪽)
ax.set_xlabel('Key (참조 대상)')  # x축 라벨
ax.set_ylabel('Query (주의를 기울이는 주체)')  # y축 라벨
ax.set_title('Scaled Dot-Product Attention 가중치 히트맵')  # 제목

# 각 셀에 수치 표시
for i in range(seq_len):  # 행 순회
    for j in range(seq_len):  # 열 순회
        val = attn_weights[0, i, j].item()  # (i, j) 위치의 가중치 값
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',  # 셀 중앙에 텍스트
                color='white' if val > 0.5 else 'black')  # 배경이 진하면 흰색 글씨

plt.colorbar(im, ax=ax)  # 색상 막대 추가
plt.tight_layout()  # 레이아웃 자동 조정
plt.show()  # 그래프 출력

print('읽는 법: 행(Query)이 열(Key)에 얼마나 주의를 기울이는지를 나타냅니다.')
print('예: (나는, 학교에) 칸의 값이 0.4이면, "나는"이 "학교에"에 40%의 주의를 기울인다는 뜻입니다.')

### 스케일링의 효과 확인: $\sqrt{d_k}$가 없으면 어떻게 될까?

아래 코드에서 $d_k = 512$인 경우 스케일링 유무에 따라 softmax 출력이 얼마나 달라지는지 확인합니다.

In [ ]:
# === 스케일링 유무에 따른 softmax 출력 비교 ===

# 큰 차원에서의 비교 (d_k = 512)
d_k_large = 512  # 실제 Transformer에서 사용하는 큰 차원
q_large = torch.randn(1, 4, d_k_large)  # 큰 차원의 Query
k_large = torch.randn(1, 4, d_k_large)  # 큰 차원의 Key

# 스케일링 없이 내적
scores_no_scale = torch.bmm(q_large, k_large.transpose(1, 2))  # Q·K^T (스케일링 없음)
attn_no_scale = F.softmax(scores_no_scale, dim=-1)  # softmax 적용

# 스케일링 적용한 내적
scores_scaled = scores_no_scale / math.sqrt(d_k_large)  # Q·K^T / sqrt(512)
attn_scaled = F.softmax(scores_scaled, dim=-1)  # softmax 적용

print('=== 스케일링 효과 비교 (d_k = 512) ===')
print(f'\n[스케일링 없음] Attention 가중치의 최대값: {attn_no_scale.max().item():.4f}')  # 거의 1에 가까움
print(f'[스케일링 없음] Attention 가중치의 최소값: {attn_no_scale.min().item():.6f}')  # 거의 0에 가까움
print(f'[스케일링 없음] 첫 행: {attn_no_scale[0, 0].detach().numpy().round(4)}')  # 거의 one-hot

print(f'\n[스케일링 있음] Attention 가중치의 최대값: {attn_scaled.max().item():.4f}')  # 적당한 값
print(f'[스케일링 있음] Attention 가중치의 최소값: {attn_scaled.min().item():.4f}')  # 적당한 값
print(f'[스케일링 있음] 첫 행: {attn_scaled[0, 0].detach().numpy().round(4)}')  # 골고루 분포

print(f'\n결론: 스케일링이 없으면 softmax 출력이 거의 one-hot이 되어')
print(f'  → 특정 위치에만 주의가 집중되고, gradient가 소실되어 학습이 어렵습니다.')
print(f'  → sqrt(d_k)로 나누면 확률이 골고루 분포되어 안정적인 학습이 가능합니다.')

## 2.2 Multi-Head Attention(다중 헤드 주의) 구현

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention(다중 헤드 주의) 구현.

    수식: MultiHead(Q, K, V) = Concat(head_1, ..., head_h) W^O
    각 head_i = Attention(Q W_i^Q, K W_i^K, V W_i^V)

    쉽게 말하면: 여러 전문가(Head)가 각자 다른 관점에서 문장을 분석한 뒤,
    결과를 합쳐서 종합적인 이해를 만드는 것입니다.

    Args:
        d_model: 모델 차원 (임베딩 차원)
        n_heads: Attention Head의 수
    """

    def __init__(self, d_model: int, n_heads: int):
        """Multi-Head Attention 초기화.

        Args:
            d_model: 모델의 전체 차원 (예: 512)
            n_heads: Head의 수 (예: 8)
        """
        super().__init__()  # nn.Module 초기화

        # d_model이 n_heads로 나누어 떨어지는지 확인
        # 왜? → 차원을 Head 수로 균등 분할해야 하기 때문
        assert d_model % n_heads == 0, f'd_model({d_model})이 n_heads({n_heads})로 나누어 떨어져야 합니다'

        self.d_model = d_model  # 전체 모델 차원 (예: 512)
        self.n_heads = n_heads  # Head 수 (예: 8)
        self.d_k = d_model // n_heads  # 각 Head의 차원 (예: 512/8 = 64)

        # Q, K, V 프로젝션(투영)을 위한 선형 변환 (가중치 행렬)
        # 왜 d_model → d_model인가? → 내부적으로 n_heads개의 (d_model → d_k) 행렬이 합쳐진 것
        self.w_q = nn.Linear(d_model, d_model, bias=False)  # Query 프로젝션: X → Q
        self.w_k = nn.Linear(d_model, d_model, bias=False)  # Key 프로젝션: X → K
        self.w_v = nn.Linear(d_model, d_model, bias=False)  # Value 프로젝션: X → V

        # 출력 프로젝션: Concat된 결과를 원래 차원으로 변환
        self.w_o = nn.Linear(d_model, d_model, bias=False)  # W^O: (d_model, d_model)

        # Scaled Dot-Product Attention (각 Head에서 사용)
        self.attention = ScaledDotProductAttention(
            temperature=math.sqrt(self.d_k)  # sqrt(d_k)로 스케일링
        )

    def forward(
        self,
        q: torch.Tensor,  # Query 입력: (batch, seq_len, d_model)
        k: torch.Tensor,  # Key 입력: (batch, seq_len, d_model)
        v: torch.Tensor,  # Value 입력: (batch, seq_len, d_model)
        mask: torch.Tensor = None  # 마스크: (batch, 1, seq_len)
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Multi-Head Attention 순전파.

        Returns:
            output: Multi-Head Attention 결과 (batch, seq_len, d_model)
            attn_weights: 각 Head의 Attention 가중치 (batch, n_heads, seq_len, seq_len)
        """
        batch_size = q.size(0)  # 배치 크기
        seq_len = q.size(1)  # 시퀀스 길이

        # 1단계: 선형 변환으로 Q, K, V 생성
        # (batch, seq_len, d_model) → (batch, seq_len, d_model)
        q = self.w_q(q)  # Query 프로젝션 적용
        k = self.w_k(k)  # Key 프로젝션 적용
        v = self.w_v(v)  # Value 프로젝션 적용

        # 2단계: n_heads개의 Head로 분할
        # (batch, seq_len, d_model) → (batch, seq_len, n_heads, d_k) → (batch, n_heads, seq_len, d_k)
        # 왜 transpose? → Head 차원을 배치 차원 옆으로 옮겨서, 각 Head를 독립 배치처럼 처리
        q = q.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        # 3단계: 각 Head를 독립적인 배치로 취급하여 Attention 계산
        # (batch * n_heads, seq_len, d_k) 형태로 변환
        q = q.contiguous().view(batch_size * self.n_heads, seq_len, self.d_k)
        k = k.contiguous().view(batch_size * self.n_heads, seq_len, self.d_k)
        v = v.contiguous().view(batch_size * self.n_heads, seq_len, self.d_k)

        # 마스크도 Head 수만큼 복제
        if mask is not None:
            mask = mask.unsqueeze(1).repeat(1, self.n_heads, 1, 1)  # (batch, n_heads, seq, seq)
            mask = mask.view(batch_size * self.n_heads, seq_len, seq_len)  # (batch*heads, seq, seq)

        # 4단계: Scaled Dot-Product Attention 실행
        attn_output, attn_weights = self.attention(q, k, v, mask)  # 각 Head별 Attention

        # 5단계: Head들을 다시 합침 (Concat)
        # (batch * n_heads, seq_len, d_k) → (batch, n_heads, seq_len, d_k)
        attn_output = attn_output.view(batch_size, self.n_heads, seq_len, self.d_k)
        # (batch, n_heads, seq_len, d_k) → (batch, seq_len, n_heads * d_k) = (batch, seq_len, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        # 6단계: 출력 프로젝션 (W^O) - 여러 Head의 결과를 하나로 통합
        output = self.w_o(attn_output)  # (batch, seq_len, d_model)

        # Attention 가중치도 보기 좋게 변환
        attn_weights = attn_weights.view(batch_size, self.n_heads, seq_len, seq_len)

        return output, attn_weights

In [ ]:
# === Multi-Head Attention 동작 확인 ===

# 하이퍼파라미터
d_model = 32  # 모델 차원 (예제이므로 작게 설정)
n_heads = 4  # 4개의 Attention Head (각 Head는 32/4 = 8차원)
seq_len = 4  # 시퀀스 길이
batch_size = 1  # 배치 크기

# 입력 생성 (Self-Attention이므로 Q=K=V=X)
x = torch.randn(batch_size, seq_len, d_model)  # (1, 4, 32) 랜덤 입력

# Multi-Head Attention 모듈 생성
mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)  # 32차원, 4개 Head

# 순전파 실행 (Self-Attention: Q=K=V=X)
output, attn_weights = mha(q=x, k=x, v=x)  # 자기 자신에 대한 Attention

print(f'입력 형태: {x.shape}')  # (1, 4, 32)
print(f'출력 형태: {output.shape}')  # (1, 4, 32) → 입력과 같은 형태!
print(f'Attention 가중치 형태: {attn_weights.shape}')  # (1, 4, 4, 4) → (batch, heads, seq, seq)
print(f'd_k (각 Head의 차원): {mha.d_k}')  # 32 / 4 = 8
print(f'\n각 Head별 Attention 가중치 합 (모두 1.0이어야 함):')  # softmax 검증
for h in range(n_heads):  # 각 Head 순회
    row_sums = attn_weights[0, h].sum(dim=-1).detach().numpy()  # 각 행의 합
    print(f'  Head {h}: {row_sums.round(4)}')  # 모두 1.0 확인

In [ ]:
# === 각 Head의 Attention 패턴 시각화 ===

tokens = ['나는', '학교에', '매일', '간다']  # 토큰 이름

fig, axes = plt.subplots(1, n_heads, figsize=(4 * n_heads, 4))  # Head 수만큼 서브플롯 생성

for h in range(n_heads):  # 각 Head 순회
    ax = axes[h]  # h번째 서브플롯
    weights = attn_weights[0, h].detach().numpy()  # h번째 Head의 가중치 (4, 4)

    im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=1)  # 히트맵
    ax.set_xticks(range(seq_len))  # x축 눈금
    ax.set_yticks(range(seq_len))  # y축 눈금
    ax.set_xticklabels(tokens, fontsize=9)  # x축 토큰 이름
    ax.set_yticklabels(tokens, fontsize=9)  # y축 토큰 이름
    ax.set_title(f'Head {h}')  # Head 번호

    # 각 셀에 수치 표시
    for i in range(seq_len):  # 행 순회
        for j in range(seq_len):  # 열 순회
            val = weights[i, j]  # 가중치 값
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    color='white' if val > 0.5 else 'black', fontsize=8)

plt.suptitle('Multi-Head Attention: 각 Head별 Attention 패턴', fontsize=14)  # 전체 제목
plt.tight_layout()  # 레이아웃 조정
plt.show()  # 그래프 출력
print('관찰: 각 Head가 서로 다른 관계 패턴을 학습합니다!')
print('  예: Head 0은 "나는"에 집중할 수 있고, Head 1은 "간다"에 집중할 수 있습니다.')

## 2.3 Positional Encoding(위치 인코딩) 구현

In [ ]:
class PositionalEncoding(nn.Module):
    """사인/코사인 Positional Encoding(위치 인코딩) 구현.

    수식:
        PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
        PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

    쉽게 말하면: 각 위치에 고유한 "위치 지문"을 만들어 임베딩에 더해줍니다.
    낮은 차원은 주기가 짧아 가까운 위치를 구분하고,
    높은 차원은 주기가 길어 먼 위치 관계를 표현합니다.

    Args:
        d_model: 모델 차원
        max_len: 지원할 최대 시퀀스 길이
    """

    def __init__(self, d_model: int, max_len: int = 5000):
        """Positional Encoding 초기화.

        Args:
            d_model: 임베딩 차원 (예: 512)
            max_len: 최대 시퀀스 길이 (기본 5000)
        """
        super().__init__()  # nn.Module 초기화

        # PE 행렬을 미리 계산 (학습하지 않는 고정 값)
        pe = torch.zeros(max_len, d_model)  # (max_len, d_model) 크기의 영행렬

        # pos: 각 위치 (0, 1, 2, ..., max_len-1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)

        # div_term: 10000^(2i/d_model)의 역수
        # 왜 log 공간에서 계산하나? → 수치 안정성 (매우 큰 수의 거듭제곱을 피함)
        # 10000^(2i/d_model) = exp(2i * log(10000) / d_model)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()  # 짝수 인덱스: 0, 2, 4, ...
            * (-math.log(10000.0) / d_model)  # -log(10000) / d_model을 곱함
        )  # (d_model/2,) → 각 차원 쌍의 주기를 결정하는 값

        # 짝수 차원: sin 적용
        pe[:, 0::2] = torch.sin(position * div_term)  # PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
        # 홀수 차원: cos 적용
        pe[:, 1::2] = torch.cos(position * div_term)  # PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

        # (max_len, d_model) → (1, max_len, d_model) 배치 차원 추가
        pe = pe.unsqueeze(0)  # 배치 차원 추가

        # 학습하지 않는 버퍼로 등록 (모델 파라미터가 아님)
        # 왜 register_buffer? → state_dict에는 포함되지만 gradient 계산은 안 함
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """입력 임베딩에 Positional Encoding을 더합니다.

        Args:
            x: 입력 임베딩 (batch, seq_len, d_model)

        Returns:
            입력 + Positional Encoding (batch, seq_len, d_model)
        """
        seq_len = x.size(1)  # 현재 시퀀스 길이
        # 시퀀스 길이만큼만 PE를 잘라서 더함
        return x + self.pe[:, :seq_len, :]  # 브로드캐스팅으로 배치 전체에 적용

In [ ]:
# === Positional Encoding 시각화 ===

d_model_vis = 64  # 시각화를 위한 차원 (작게 설정)
max_len_vis = 100  # 최대 100개 위치

# Positional Encoding 생성
pe_module = PositionalEncoding(d_model=d_model_vis, max_len=max_len_vis)  # PE 모듈 생성
pe_values = pe_module.pe[0, :max_len_vis, :].numpy()  # (100, 64) PE 행렬 추출

# 히트맵으로 전체 PE 행렬 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # 2개의 서브플롯

# 서브플롯 1: PE 행렬 전체 히트맵
im = axes[0].imshow(pe_values, cmap='RdBu', aspect='auto')  # 빨강-파랑 색상표
axes[0].set_xlabel('차원 (i)')  # x축: 차원 인덱스
axes[0].set_ylabel('위치 (pos)')  # y축: 위치
axes[0].set_title('Positional Encoding 행렬')  # 제목
plt.colorbar(im, ax=axes[0])  # 색상 막대

# 서브플롯 2: 특정 차원의 PE 값을 위치에 따라 그래프로
dims_to_plot = [0, 1, 4, 5, 10, 11]  # 시각화할 차원들 (짝수=sin, 홀수=cos)
for d in dims_to_plot:  # 각 차원 순회
    label = f'dim={d} ({"sin" if d % 2 == 0 else "cos"})'  # 짝수면 sin, 홀수면 cos
    axes[1].plot(pe_values[:, d], label=label, alpha=0.7)  # 위치에 따른 PE 값

axes[1].set_xlabel('위치 (pos)')  # x축: 위치
axes[1].set_ylabel('PE 값')  # y축: PE 값
axes[1].set_title('차원별 PE 값 변화')  # 제목
axes[1].legend(fontsize=8)  # 범례
axes[1].set_xlim(0, max_len_vis)  # x축 범위

plt.suptitle('Positional Encoding 시각화', fontsize=14)  # 전체 제목
plt.tight_layout()  # 레이아웃 조정
plt.show()  # 그래프 출력

print('관찰 1: 낮은 차원(dim=0,1)은 주기가 짧고, 높은 차원(dim=10,11)은 주기가 깁니다!')
print('관찰 2: 이를 통해 가까운 위치와 먼 위치를 동시에 인코딩할 수 있습니다.')
print('관찰 3: 히트맵에서 세로줄 패턴이 보이는데, 이것이 각 위치의 고유한 "지문"입니다.')

## 2.4 Transformer Encoder Block(인코더 블록) 구현

In [ ]:
class FeedForwardNetwork(nn.Module):
    """Position-wise Feed-Forward Network(위치별 순방향 신경망) 구현.

    수식: FFN(x) = ReLU(x W_1 + b_1) W_2 + b_2

    쉽게 말하면: Attention이 "어디를 볼지" 결정했다면,
    FFN은 "본 것을 어떻게 처리할지" 결정합니다.
    차원을 4배로 늘렸다가 다시 줄이는 병목 구조입니다.

    Args:
        d_model: 입력/출력 차원
        d_ff: 은닉층 차원 (보통 d_model의 4배)
        dropout: 드롭아웃(무작위 비활성화) 비율
    """

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        """FFN 초기화.

        Args:
            d_model: 모델 차원 (예: 512)
            d_ff: FFN 은닉층 차원 (예: 2048)
            dropout: 드롭아웃 비율 (예: 0.1)
        """
        super().__init__()  # nn.Module 초기화

        # 첫 번째 선형 변환: d_model → d_ff (차원 확장)
        # 왜 확장하나? → 더 넓은 공간에서 비선형 패턴을 학습하기 위해
        self.linear1 = nn.Linear(d_model, d_ff)  # W_1: (d_model, d_ff)
        # 두 번째 선형 변환: d_ff → d_model (차원 축소, 원래 차원 복원)
        self.linear2 = nn.Linear(d_ff, d_model)  # W_2: (d_ff, d_model)
        # 드롭아웃: 과적합(overfitting) 방지
        # 왜 드롭아웃? → 학습 시 일부 뉴런을 랜덤하게 꺼서 모델이 특정 뉴런에 의존하지 않게 함
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """FFN 순전파.

        Args:
            x: 입력 (batch, seq_len, d_model)

        Returns:
            출력 (batch, seq_len, d_model)
        """
        # x W_1 + b_1: (batch, seq_len, d_model) → (batch, seq_len, d_ff)
        out = self.linear1(x)  # 첫 번째 선형 변환
        # ReLU 활성화: 음수를 0으로 → 비선형성 추가
        out = F.relu(out)
        # 드롭아웃 적용
        out = self.dropout(out)
        # ReLU(x W_1 + b_1) W_2 + b_2: (batch, seq_len, d_ff) → (batch, seq_len, d_model)
        out = self.linear2(out)  # 두 번째 선형 변환 (차원 복원)
        return out  # (batch, seq_len, d_model)

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """Transformer Encoder Block(인코더 블록) 구현.

    구조: Self-Attention → Add & Norm → FFN → Add & Norm

    쉽게 말하면: "문장을 읽고(Attention) → 메모를 유지하며(Residual) →
    정리하고(LayerNorm) → 깊이 생각하고(FFN) → 다시 메모를 유지하며(Residual) →
    정리하는(LayerNorm)" 과정을 N번 반복합니다.

    Args:
        d_model: 모델 차원
        n_heads: Attention Head 수
        d_ff: FFN 은닉층 차원
        dropout: 드롭아웃 비율
    """

    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        """Encoder Block 초기화.

        Args:
            d_model: 모델 차원 (예: 512)
            n_heads: Head 수 (예: 8)
            d_ff: FFN 차원 (예: 2048)
            dropout: 드롭아웃 비율 (예: 0.1)
        """
        super().__init__()  # nn.Module 초기화

        # 서브레이어 1: Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, n_heads)
        # 서브레이어 2: Position-wise Feed-Forward Network
        self.feed_forward = FeedForwardNetwork(d_model, d_ff, dropout)

        # Layer Normalization(층 정규화) - 각 서브레이어 후에 적용
        self.norm1 = nn.LayerNorm(d_model)  # Self-Attention 후 정규화
        self.norm2 = nn.LayerNorm(d_model)  # FFN 후 정규화

        # 드롭아웃 (각 서브레이어 출력에 적용)
        self.dropout1 = nn.Dropout(dropout)  # Attention 출력 드롭아웃
        self.dropout2 = nn.Dropout(dropout)  # FFN 출력 드롭아웃

    def forward(
        self,
        x: torch.Tensor,  # 입력: (batch, seq_len, d_model)
        mask: torch.Tensor = None  # 마스크 (선택적)
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Encoder Block 순전파.

        Returns:
            output: Block 출력 (batch, seq_len, d_model)
            attn_weights: Attention 가중치 (batch, n_heads, seq_len, seq_len)
        """
        # === 서브레이어 1: Multi-Head Self-Attention + Add & Norm ===
        # Self-Attention: Q=K=V=x (자기 자신에 대한 Attention)
        attn_output, attn_weights = self.self_attention(x, x, x, mask)  # Attention 계산
        attn_output = self.dropout1(attn_output)  # 드롭아웃 적용
        # Residual Connection(잔차 연결) + Layer Normalization
        # 왜 x를 더하나? → 원래 입력 정보를 유지하면서, Attention이 학습한 "변화량"만 추가
        x = self.norm1(x + attn_output)  # Add & Norm

        # === 서브레이어 2: Feed-Forward Network + Add & Norm ===
        ff_output = self.feed_forward(x)  # FFN 적용
        ff_output = self.dropout2(ff_output)  # 드롭아웃 적용
        # Residual Connection + Layer Normalization
        x = self.norm2(x + ff_output)  # Add & Norm

        return x, attn_weights

In [ ]:
# === Transformer Encoder Block 동작 확인 ===

# 하이퍼파라미터
d_model = 32  # 모델 차원
n_heads = 4  # Attention Head 수
d_ff = 128  # FFN 은닉층 차원 (d_model의 4배)
seq_len = 4  # 시퀀스 길이
batch_size = 2  # 배치 크기

# Encoder Block 생성
encoder_block = TransformerEncoderBlock(
    d_model=d_model,  # 32차원
    n_heads=n_heads,  # 4개 Head
    d_ff=d_ff,  # FFN 128차원
    dropout=0.1  # 10% 드롭아웃
)

# 입력 생성 (임베딩 + Positional Encoding 후의 입력이라고 가정)
x = torch.randn(batch_size, seq_len, d_model)  # (2, 4, 32) 랜덤 입력

# 순전파 실행
encoder_block.eval()  # 평가 모드 (드롭아웃 비활성화)
output, attn_weights = encoder_block(x)  # Encoder Block 통과

print(f'입력 형태: {x.shape}')  # (2, 4, 32)
print(f'출력 형태: {output.shape}')  # (2, 4, 32) → 입력과 동일한 형태!
print(f'Attention 가중치 형태: {attn_weights.shape}')  # (2, 4, 4, 4)

# 파라미터 수 계산
total_params = sum(p.numel() for p in encoder_block.parameters())  # 전체 파라미터 수
print(f'\nEncoder Block 파라미터 수: {total_params:,}개')

# 각 서브모듈별 파라미터 수
for name, module in encoder_block.named_children():  # 자식 모듈 순회
    params = sum(p.numel() for p in module.parameters())  # 모듈별 파라미터 수
    print(f'  {name}: {params:,}개')

## 2.5 전체 Transformer Encoder 조립

In [ ]:
class TransformerEncoder(nn.Module):
    """전체 Transformer Encoder(인코더) 구현.

    구조: Token Embedding + Positional Encoding → N개의 Encoder Block

    쉽게 말하면: 단어를 벡터로 변환하고(Embedding), 위치 정보를 추가한 뒤(PE),
    Encoder Block을 N번 통과시켜 문맥을 깊이 이해한 표현을 만듭니다.

    Args:
        vocab_size: 어휘 사전 크기
        d_model: 모델 차원
        n_heads: Attention Head 수
        d_ff: FFN 은닉층 차원
        n_layers: Encoder Block 수
        max_len: 최대 시퀀스 길이
        dropout: 드롭아웃 비율
    """

    def __init__(
        self,
        vocab_size: int,  # 어휘 크기
        d_model: int,  # 모델 차원
        n_heads: int,  # Head 수
        d_ff: int,  # FFN 차원
        n_layers: int,  # Encoder Block 수
        max_len: int = 512,  # 최대 시퀀스 길이
        dropout: float = 0.1  # 드롭아웃 비율
    ):
        """Transformer Encoder 초기화."""
        super().__init__()  # nn.Module 초기화

        # 토큰 임베딩(Token Embedding): 정수 토큰 ID → d_model 차원 벡터
        self.token_embedding = nn.Embedding(vocab_size, d_model)  # 어휘 크기 × d_model
        # Positional Encoding: 위치 정보 추가
        self.positional_encoding = PositionalEncoding(d_model, max_len)  # 사인/코사인 PE
        # 드롭아웃: 임베딩 후 적용
        self.dropout = nn.Dropout(dropout)

        # N개의 Encoder Block을 ModuleList로 쌓기
        # 왜 ModuleList? → 각 Block이 독립적인 파라미터를 갖도록 하기 위해
        self.encoder_blocks = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)  # n_layers개의 Block
        ])

        # 임베딩 스케일링 팩터
        self.d_model = d_model  # sqrt(d_model)을 곱하기 위해 저장

    def forward(
        self,
        x: torch.Tensor,  # 입력 토큰 ID: (batch, seq_len)
        mask: torch.Tensor = None  # 마스크 (선택적)
    ) -> tuple[torch.Tensor, list[torch.Tensor]]:
        """Transformer Encoder 순전파.

        Returns:
            output: 최종 인코딩 (batch, seq_len, d_model)
            all_attn_weights: 각 Block의 Attention 가중치 리스트
        """
        # 1단계: 토큰 임베딩
        # (batch, seq_len) → (batch, seq_len, d_model)
        x = self.token_embedding(x)  # 정수 ID를 벡터로 변환

        # 임베딩에 sqrt(d_model)을 곱함 (논문 Section 3.4)
        # 왜? → PE 값(-1~1)과 임베딩의 스케일을 맞추기 위함
        x = x * math.sqrt(self.d_model)

        # 2단계: Positional Encoding 추가
        x = self.positional_encoding(x)  # 위치 정보를 더함
        x = self.dropout(x)  # 드롭아웃 적용

        # 3단계: N개의 Encoder Block 순차 통과
        all_attn_weights = []  # 각 Block의 Attention 가중치 저장
        for block in self.encoder_blocks:  # 각 Block 순회
            x, attn_weights = block(x, mask)  # Block 통과
            all_attn_weights.append(attn_weights)  # 가중치 저장

        return x, all_attn_weights

In [ ]:
# === 전체 Transformer Encoder 동작 확인 ===

# 하이퍼파라미터 (작은 규모의 모델)
vocab_size = 1000  # 어휘 크기 (1000개의 토큰)
d_model = 64  # 모델 차원
n_heads = 4  # 4개의 Attention Head
d_ff = 256  # FFN 은닉층 차원 (d_model × 4)
n_layers = 3  # 3개의 Encoder Block
seq_len = 10  # 시퀀스 길이
batch_size = 2  # 배치 크기

# Transformer Encoder 생성
encoder = TransformerEncoder(
    vocab_size=vocab_size,  # 어휘 1000개
    d_model=d_model,  # 64차원
    n_heads=n_heads,  # 4 Head
    d_ff=d_ff,  # FFN 256차원
    n_layers=n_layers  # 3층
)

# 입력 생성 (정수 토큰 ID)
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))  # (2, 10) 랜덤 토큰

# 순전파
encoder.eval()  # 평가 모드
output, all_attn = encoder(input_ids)  # Encoder 통과

print(f'입력 토큰 형태: {input_ids.shape}')  # (2, 10)
print(f'출력 형태: {output.shape}')  # (2, 10, 64)
print(f'Encoder Block 수: {len(all_attn)}')  # 3
print(f'각 Block의 Attention 가중치 형태: {all_attn[0].shape}')  # (2, 4, 10, 10)

# 전체 파라미터 수
total_params = sum(p.numel() for p in encoder.parameters())
print(f'\n전체 파라미터 수: {total_params:,}개')

# 원본 Transformer (Base)와 비교
print(f'\n참고: 원본 Transformer Base의 파라미터 수: ~65M개')
print(f'→ 우리 모델({total_params:,}개)은 학습용으로 매우 작게 만든 것입니다.')

## 2.6 Causal Mask(인과적 마스크) 구현 및 확인

Decoder에서 미래 토큰을 볼 수 없도록 차단하는 마스크입니다.

In [ ]:
def create_causal_mask(seq_len: int) -> torch.Tensor:
    """Causal Mask(인과적 마스크) 생성.

    미래 위치를 볼 수 없도록 상삼각 부분을 마스킹합니다.

    쉽게 말하면: 시험에서 아직 나오지 않은 문제의 답을 미리 볼 수 없도록
    가리개를 만드는 것입니다.

    Args:
        seq_len: 시퀀스 길이

    Returns:
        mask: (1, seq_len, seq_len) 크기의 마스크 텐서
               1 = 볼 수 있음, 0 = 볼 수 없음 (마스킹)
    """
    # 하삼각 행렬 생성: 대각선 포함 아래쪽만 1, 나머지 0
    mask = torch.tril(torch.ones(seq_len, seq_len))  # (seq_len, seq_len) 하삼각 행렬
    # 배치 차원 추가
    mask = mask.unsqueeze(0)  # (1, seq_len, seq_len)
    return mask  # 1이면 참조 가능, 0이면 마스킹


# === Causal Mask 동작 확인 ===

seq_len = 5  # 5개의 토큰
causal_mask = create_causal_mask(seq_len)  # 마스크 생성

print('Causal Mask (1=참조 가능, 0=마스킹):')
print(causal_mask[0].numpy())
print()

# 마스크 적용 후 Attention 비교
tokens = ['BOS', '나는', '학교에', '매일', '간다']  # 5개 토큰
d_k = 8  # 차원
q = torch.randn(1, seq_len, d_k)  # Query
k = torch.randn(1, seq_len, d_k)  # Key
v = torch.randn(1, seq_len, d_k)  # Value

attention = ScaledDotProductAttention(math.sqrt(d_k))  # Attention 모듈

# 마스크 없이 (양방향) - Encoder 방식
_, attn_bidirectional = attention(q, k, v, mask=None)  # 전체 참조 가능
# 마스크 적용 (단방향) - Decoder 방식
_, attn_causal = attention(q, k, v, mask=causal_mask)  # 미래 마스킹

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # 2개 서브플롯

for ax, weights, title in [
    (axes[0], attn_bidirectional, 'Bidirectional (Encoder, 양방향)'),
    (axes[1], attn_causal, 'Causal/Masked (Decoder, 단방향)')
]:
    w = weights[0].detach().numpy()
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(tokens, fontsize=9)
    ax.set_yticklabels(tokens, fontsize=9)
    ax.set_title(title)
    for i in range(seq_len):
        for j in range(seq_len):
            val = w[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    color='white' if val > 0.5 else 'black', fontsize=8)
    plt.colorbar(im, ax=ax)

plt.suptitle('Causal Mask 효과: Encoder(양방향) vs Decoder(단방향)', fontsize=13)
plt.tight_layout()
plt.show()
print('관찰: Decoder(오른쪽)에서는 상삼각 부분이 0이 되어 미래 토큰을 참조할 수 없습니다!')
print('  예: "나는"(2행)은 "학교에", "매일", "간다"의 가중치가 모두 0입니다.')

---
# Part 3: BERT 이해 & 활용

## 3.1 BERT란?

**BERT** = **B**idirectional **E**ncoder **R**epresentations from **T**ransformers
(양방향 인코더 표현 기반 트랜스포머)

2018년 Google이 발표한 모델로, Transformer의 **Encoder만** 사용합니다.

### 핵심 아이디어: 양방향 문맥 이해

기존 모델들의 한계:
- **GPT (2018)**: 왼쪽 → 오른쪽으로만 읽음 (단방향)
- **ELMo (2018)**: 왼쪽→오른쪽 + 오른쪽→왼쪽을 따로 학습 후 합침 (얕은 양방향)

BERT의 혁신:
- **진정한 양방향**: 모든 위치가 모든 위치를 동시에 참조 (깊은 양방향)
- 이를 가능하게 한 것이 바로 **Masked Language Model(마스크 언어 모델, MLM)** 학습법!

> **비유**: GPT는 소설을 앞에서부터 순서대로 읽는 독자이고, BERT는 빈칸 채우기 시험을 보는 학생입니다. "나는 ___에 간다"라는 문제에서 BERT는 앞("나는")과 뒤("간다")를 모두 보고 "학교"를 맞출 수 있습니다.

## 3.2 BERT의 사전학습 (Pre-training)

BERT는 2가지 과제로 동시에 사전학습됩니다:

### 과제 1: MLM (Masked Language Model, 마스크 언어 모델) - 마스크된 단어 예측

**쉽게 말하면**: 빈칸 채우기 시험입니다. 문장에서 일부 단어를 가리고 맞추게 합니다.

**방법**: 입력 토큰의 15%를 랜덤하게 선택하여:
- 80%는 `[MASK]` 토큰으로 치환
- 10%는 랜덤한 다른 토큰으로 치환
- 10%는 그대로 유지

**예시**:
```
원본: "나는 학교에 매일 간다"
입력: "나는 [MASK] 매일 간다"
목표: [MASK] 위치에서 "학교에"를 예측
```

**왜 이렇게 복잡하게?**
- 80% MASK: 모델이 마스크된 위치를 예측하도록 학습
- 10% 랜덤: Fine-tuning(미세조정) 시에는 MASK 토큰이 없으므로 불일치 완화
- 10% 유지: 모델이 올바른 토큰도 잘 표현하도록 학습

### 과제 2: NSP (Next Sentence Prediction, 다음 문장 예측) - 다음 문장 예측

**쉽게 말하면**: 두 문장이 원래 이어지는 문장인지 아닌지 맞추는 것입니다.

**방법**: 두 문장 A, B를 입력하여 B가 A의 실제 다음 문장인지 예측

```
입력: "[CLS] 나는 학교에 간다 [SEP] 수업이 재미있다 [SEP]"
라벨: IsNext (50%) 또는 NotNext (50%)
```

- `[CLS]`: Classification(분류) 토큰 - 문장 쌍의 관계를 나타내는 특수 토큰
- `[SEP]`: Separator(구분) 토큰 - 두 문장의 경계를 나타내는 특수 토큰

**목적**: 문장 간 관계 이해 (질의응답, 자연어 추론 등에 활용)

## 3.3 BERT 입력 구조: 3가지 Embedding(임베딩)의 합

BERT의 입력은 3가지 Embedding을 더한 것입니다:

```
입력:    [CLS]  나는  학교에  간다  [SEP]  수업이  재미있다  [SEP]
         ──────────────────────────────────────────────────────────
Token:   E_CLS  E_나  E_학교  E_간  E_SEP  E_수업  E_재미   E_SEP
         +      +     +      +     +      +       +        +
Segment: E_A    E_A   E_A    E_A   E_A    E_B     E_B      E_B
         +      +     +      +     +      +       +        +
Position:E_0    E_1   E_2    E_3   E_4    E_5     E_6      E_7
         ──────────────────────────────────────────────────────────
         = 최종 입력 임베딩 (각 위치별로 3개의 벡터를 더함)
```

**쉽게 말하면**: 각 토큰은 "단어의 뜻(Token) + 어느 문장인지(Segment) + 몇 번째인지(Position)" 세 가지 정보를 합쳐서 표현됩니다.

| Embedding | 설명 | 학습 방식 |
|-----------|------|----------|
| **Token Embedding** | 각 토큰의 의미 표현 | 학습 (lookup table) |
| **Segment Embedding** | 문장 A(0) vs 문장 B(1) 구분 | 학습 |
| **Position Embedding** | 위치 정보 (0, 1, 2, ...) | **학습** (원본 Transformer의 sin/cos과 다름!) |

### BERT vs 원본 Transformer의 Positional Encoding 차이

| | 원본 Transformer | BERT |
|--|---|---|
| 방식 | sin/cos 고정 함수 | **학습 가능한 임베딩** |
| 파라미터 | 0 (고정) | max_len × d_model (학습) |
| 외삽 | 학습 범위 밖도 가능 | 학습 범위 내만 |
| 최대 길이 | 이론상 무한 | **512** (BERT의 제약) |

## 3.4 Fine-tuning(미세조정): 사전학습 → 태스크 적응

BERT의 사용 방식은 2단계입니다:

```
1단계: Pre-training (사전학습)     2단계: Fine-tuning (미세조정)
───────────────────────           ─────────────────────────
대량의 텍스트 (위키피디아 등)       소량의 라벨 데이터
MLM + NSP 학습                    태스크별 분류 헤드 추가
범용적 언어 이해 학습               특정 태스크에 적응
수백 GPU, 수 일                    1 GPU, 수 시간
```

**쉽게 말하면**: Pre-training은 "학교에서 기초 교육을 받는 것"이고, Fine-tuning은 "직장에서 실무에 적응하는 것"입니다. 기초 교육(Pre-training)이 탄탄하면 어떤 직장(태스크)이든 빠르게 적응할 수 있습니다.

### Fine-tuning 방법 (태스크별)

| 태스크 | 입력 | 출력 위치 | 예시 |
|--------|------|-----------|------|
| 문장 분류 | `[CLS] 문장 [SEP]` | `[CLS]` 벡터 → 분류기 | 감정 분석, 스팸 탐지 |
| 문장 쌍 분류 | `[CLS] 문장A [SEP] 문장B [SEP]` | `[CLS]` 벡터 → 분류기 | 자연어 추론, 의미 유사도 |
| 토큰 분류 | `[CLS] 토큰들 [SEP]` | 각 토큰 벡터 → 분류기 | NER (Named Entity Recognition, 개체명 인식) |
| 질의응답 | `[CLS] 질문 [SEP] 문서 [SEP]` | 문서 내 시작/끝 위치 | SQuAD |

핵심: **사전학습된 가중치를 초기값으로 사용**하고, 적은 데이터로 빠르게 태스크에 적응!

## 3.5 DistilBERT로 감정 분류 실습

이제 실제로 BERT 계열 모델을 사용하여 감정 분류를 해보겠습니다.

**DistilBERT**: BERT를 Knowledge Distillation(지식 증류)으로 경량화한 모델
- BERT 대비 40% 작음, 60% 빠름, 성능 97% 유지
- 6층 (BERT는 12층)

> **비유**: Knowledge Distillation은 "베테랑 선생님(BERT)이 핵심 내용만 정리해서 학생(DistilBERT)에게 가르치는 것"과 같습니다. 학생은 선생님보다 작지만, 핵심은 거의 다 알고 있습니다.

### 3.5.1 토크나이저(Tokenizer) 이해

In [ ]:
from transformers import AutoTokenizer  # HuggingFace 토크나이저 자동 로드

# DistilBERT 토크나이저 로드
# 왜 uncased? → 대소문자를 구분하지 않는 모델 (Hello = hello)
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# === 토크나이저 상세 설명 ===

# 1. 기본 토큰화
text = "I love studying deep learning!"  # 영어 예시 문장
tokens = tokenizer.tokenize(text)  # 텍스트 → 토큰 리스트
print(f'원문: {text}')
print(f'토큰화: {tokens}')

# 2. 토큰 → ID 변환 (컴퓨터는 문자열이 아닌 숫자를 처리)
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(f'토큰 ID: {token_ids}')  # 어휘 사전에서의 인덱스

# 3. 전체 인코딩 (특수 토큰 포함)
encoded = tokenizer(
    text,  # 입력 텍스트
    return_tensors='pt',  # PyTorch 텐서로 반환
    padding=True,  # 패딩(빈 공간 채우기) 추가
    truncation=True,  # 최대 길이 초과 시 잘라냄
    max_length=32  # 최대 32토큰
)

print(f'\n전체 인코딩 결과:')
print(f'  input_ids: {encoded["input_ids"]}')  # [CLS] + 토큰들 + [SEP] + [PAD]...
print(f'  attention_mask: {encoded["attention_mask"]}')  # 1=실제 토큰, 0=패딩

# 4. ID → 토큰으로 복원
decoded_tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
print(f'  토큰 복원: {decoded_tokens}')  # [CLS], 토큰들, [SEP]

# 5. 어휘 사전 크기
print(f'\n어휘 사전 크기: {tokenizer.vocab_size:,}개')  # 30,522개 (BERT 기본)
print(f'특수 토큰: {tokenizer.special_tokens_map}')  # [UNK], [SEP], [PAD], [CLS], [MASK]

In [ ]:
# === 서브워드(Subword) 토크나이저의 동작 원리 ===

# BERT는 WordPiece 토크나이저를 사용합니다.
# 쉽게 말하면: 자주 등장하는 단어는 그대로 유지하고,
# 드문 단어는 작은 조각(서브워드)으로 분할합니다.
# 이렇게 하면 어떤 단어든 처리할 수 있습니다! (미등록어 문제 해결)

examples = [  # 다양한 예시
    "Transformer is amazing",  # 일반적인 단어들
    "Tokenization is important",  # 'tokenization'이 서브워드로 분할될 수 있음
    "I am unhappy about this",  # 'unhappy' → 'un' + '##happy'
    "The cryptocurrency market crashed",  # 'cryptocurrency' → 서브워드 분할
]

print('=== WordPiece 토크나이저 동작 예시 ===')
for text in examples:  # 각 예시 순회
    tokens = tokenizer.tokenize(text)  # 토큰화
    print(f'\n원문: "{text}"')
    print(f'토큰: {tokens}')

print('\n## 접두사의 의미: ##이 붙은 토큰은 앞 토큰에 이어지는 서브워드입니다.')
print('  예: ["un", "##happy"] → "unhappy"')
print('\n핵심: 서브워드 토크나이저 덕분에 어떤 단어든 처리 가능합니다!')
print('  → 미등록어(OOV, Out of Vocabulary) 문제 완전 해결!')

### 3.5.2 데이터셋 준비 및 Fine-tuning(미세조정)

In [ ]:
from datasets import load_dataset  # HuggingFace 데이터셋 로드

# SST-2 (Stanford Sentiment Treebank) 데이터셋 로드
# SST-2는 GLUE 벤치마크의 감정 분류 태스크입니다.
# 영화 리뷰의 긍정(1)/부정(0) 감정을 분류합니다.
dataset = load_dataset('glue', 'sst2')

print(f'데이터셋 구조: {dataset}')  # train, validation, test 분할
print(f'\n학습 데이터 수: {len(dataset["train"]):,}개')  # ~67K개
print(f'검증 데이터 수: {len(dataset["validation"]):,}개')  # ~872개

# 데이터 샘플 확인
print(f'\n학습 데이터 샘플:')
for i in range(5):  # 첫 5개
    sample = dataset['train'][i]
    label = '긍정' if sample['label'] == 1 else '부정'  # 라벨 한국어 변환
    print(f'  [{label}] {sample["sentence"]}')

In [ ]:
# === 데이터셋 토큰화 (전처리) ===

def tokenize_function(examples: dict) -> dict:
    """배치 단위로 텍스트를 토큰화하는 함수.

    Args:
        examples: 배치 데이터 (sentence 키 포함)

    Returns:
        토큰화된 결과 (input_ids, attention_mask 포함)
    """
    return tokenizer(
        examples['sentence'],  # 문장 텍스트
        padding='max_length',  # 최대 길이까지 패딩 (모든 입력을 같은 길이로 맞춤)
        truncation=True,  # 최대 길이 초과 시 잘라냄
        max_length=128  # 최대 128 토큰
    )

# 전체 데이터셋에 토큰화 적용
tokenized_datasets = dataset.map(
    tokenize_function,  # 토큰화 함수
    batched=True  # 배치 단위로 처리 (개별 처리보다 훨씬 빠름)
)

# 빠른 실습을 위해 데이터 서브셋 사용
# 왜 서브셋? → 전체 데이터(67K)로 학습하면 오래 걸리므로, 2000개만 사용
small_train = tokenized_datasets['train'].shuffle(seed=42).select(range(2000))  # 학습: 2000개
small_val = tokenized_datasets['validation'].shuffle(seed=42).select(range(500))  # 검증: 500개

print(f'학습 데이터 (서브셋): {len(small_train)}개')
print(f'검증 데이터 (서브셋): {len(small_val)}개')
print(f'\n토큰화 후 컬럼: {small_train.column_names}')  # input_ids, attention_mask 등

In [ ]:
from transformers import AutoModelForSequenceClassification  # 문장 분류 모델
from transformers import TrainingArguments, Trainer  # 학습 설정, 학습기

# === DistilBERT 감정 분류 모델 로드 ===

# 사전학습된 DistilBERT에 분류 헤드(Linear 레이어)를 추가
# 왜 num_labels=2? → 긍정/부정 2개 클래스를 분류하기 위해
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',  # 사전학습 모델 이름
    num_labels=2  # 출력 라벨 수 (긍정, 부정)
)

# 모델 구조 요약
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'전체 파라미터 수: {total_params:,}개')  # ~67M
print(f'학습 가능 파라미터: {trainable_params:,}개')  # 전부 학습 가능 (Full Fine-tuning)
print(f'\n모델 구조 (최상위):')
for name, child in model.named_children():  # 최상위 모듈 순회
    params = sum(p.numel() for p in child.parameters())
    print(f'  {name}: {params:,}개')

In [ ]:
# === 평가 지표 정의 ===

def compute_metrics(eval_pred: tuple) -> dict:
    """정확도(Accuracy)를 계산하는 함수.

    Args:
        eval_pred: (logits, labels) 튜플

    Returns:
        {'accuracy': 정확도 값}
    """
    logits, labels = eval_pred  # 모델 출력과 정답 라벨
    predictions = np.argmax(logits, axis=-1)  # logit이 가장 큰 클래스가 예측값
    accuracy = (predictions == labels).mean()  # 예측 == 정답인 비율
    return {'accuracy': accuracy}


# === 학습 설정 ===

training_args = TrainingArguments(
    output_dir='./results',  # 체크포인트 저장 디렉토리
    num_train_epochs=2,  # 에포크(epoch) 수: 데이터를 2번 반복 학습
    per_device_train_batch_size=16,  # 학습 배치 크기 (GPU 메모리에 맞게 조절)
    per_device_eval_batch_size=32,  # 평가 배치 크기 (학습보다 크게 설정 가능)
    learning_rate=2e-5,  # 학습률 (BERT Fine-tuning 권장값: 2e-5 ~ 5e-5)
    weight_decay=0.01,  # 가중치 감쇠 (과적합 방지)
    eval_strategy='epoch',  # 에포크마다 검증 데이터로 평가
    save_strategy='epoch',  # 에포크마다 체크포인트 저장
    logging_steps=50,  # 50스텝마다 로그 출력
    load_best_model_at_end=True,  # 학습 후 가장 좋은 모델 로드
    metric_for_best_model='accuracy',  # 최적 모델 기준: 정확도
    report_to='none',  # WandB 등 외부 리포팅 비활성화 (Colab에서는 필요 없음)
)

print('학습 설정 완료!')
print(f'  에포크: {training_args.num_train_epochs}')
print(f'  학습률: {training_args.learning_rate}')
print(f'  배치 크기: {training_args.per_device_train_batch_size}')

In [ ]:
# === Trainer로 Fine-tuning 실행 ===

trainer = Trainer(
    model=model,  # Fine-tuning할 모델
    args=training_args,  # 학습 설정
    train_dataset=small_train,  # 학습 데이터
    eval_dataset=small_val,  # 검증 데이터
    compute_metrics=compute_metrics,  # 평가 지표 함수
)

# 학습 시작
print('Fine-tuning 시작...')
print('(2000개 데이터, 2 에포크 → Colab GPU에서 약 2~3분 소요)')
train_result = trainer.train()  # 학습 실행
print('\nFine-tuning 완료!')

# 학습 결과
print(f'\n학습 결과:')
print(f'  총 학습 스텝: {train_result.global_step}')
print(f'  최종 학습 손실: {train_result.training_loss:.4f}')

In [ ]:
# === 검증 세트 평가 ===

eval_results = trainer.evaluate()  # 검증 데이터로 평가

print('=== 검증 결과 ===')
print(f'  정확도: {eval_results["eval_accuracy"]:.4f}')
print(f'  손실: {eval_results["eval_loss"]:.4f}')
print(f'\n해석: 2000개의 학습 데이터만으로도 높은 정확도를 달성했습니다!')
print(f'  → 이것이 사전학습(Pre-training)의 위력입니다.')
print(f'  → 이미 언어를 이해하고 있는 모델을 가져와서 소량의 데이터로 적응시켰기 때문입니다.')

In [ ]:
# === 새로운 문장으로 예측 테스트 ===

test_sentences = [  # 테스트 문장들
    "This movie is absolutely wonderful and amazing!",  # 매우 긍정적
    "I really hated this terrible film.",  # 매우 부정적
    "The acting was okay but the plot was boring.",  # 약한 부정
    "A masterpiece of modern cinema!",  # 긍정적
    "Waste of time and money.",  # 부정적
]

model.eval()  # 평가 모드로 전환 (드롭아웃 비활성화)

print('=== 감정 분류 예측 결과 ===')
for sentence in test_sentences:  # 각 문장 순회
    # 토큰화
    inputs = tokenizer(
        sentence,
        return_tensors='pt',  # PyTorch 텐서
        padding=True,
        truncation=True,
        max_length=128
    ).to(model.device)  # 모델과 같은 디바이스로

    # 예측 (gradient 계산 불필요 → 메모리 절약)
    with torch.no_grad():
        outputs = model(**inputs)  # 순전파

    # softmax로 확률 변환
    probs = F.softmax(outputs.logits, dim=-1)  # logit → 확률
    pred_label = probs.argmax(dim=-1).item()  # 가장 높은 확률의 클래스
    confidence = probs.max().item()  # 최대 확률 (신뢰도)

    # 결과 출력
    sentiment = '긍정' if pred_label == 1 else '부정'
    print(f'\n문장: "{sentence}"')
    print(f'  예측: {sentiment} (신뢰도: {confidence:.2%})')
    print(f'  확률: 부정={probs[0][0]:.2%}, 긍정={probs[0][1]:.2%}')

---
# Part 4: 정리

## 4.1 Transformer 계보: 주요 모델들

```
2017 --- Transformer ("Attention is All You Need")
           |
           +-- Encoder-only (양방향, 이해 중심)
           |     +-- 2018: BERT (Google) --- 양방향 문맥 이해
           |     +-- 2019: RoBERTa (Meta) -- BERT 학습 개선
           |     +-- 2019: ALBERT (Google) - BERT 경량화
           |     +-- 2019: DistilBERT (HF) - Knowledge Distillation
           |     +-- 2020: ViT (Google) ---- 이미지에 Transformer 적용!
           |
           +-- Decoder-only (단방향, 생성 중심)
           |     +-- 2018: GPT (OpenAI) ---- 단방향 텍스트 생성
           |     +-- 2019: GPT-2 (OpenAI) -- 더 크게, 더 강하게
           |     +-- 2020: GPT-3 (OpenAI) -- Few-shot Learning의 등장
           |     +-- 2023: GPT-4 (OpenAI) -- 멀티모달
           |     +-- 2023: LLaMA (Meta) ---- 오픈소스 LLM
           |     +-- 2024: Claude (Anthropic) -- 안전한 AI
           |
           +-- Encoder-Decoder (양방향 이해 + 단방향 생성)
                 +-- 2019: T5 (Google) ---- 모든 NLP를 Text-to-Text로
                 +-- 2019: BART (Meta) ---- 노이즈 제거 사전학습
                 +-- 2022: Whisper (OpenAI) -- 음성 인식
```

> **쉽게 말하면**: Encoder-only 계열(BERT)은 "문장을 이해하는 데 특화"되어 분류/검색에 강하고, Decoder-only 계열(GPT)은 "문장을 생성하는 데 특화"되어 대화/글쓰기에 강합니다.

## 4.2 Encoder-only vs Decoder-only vs Encoder-Decoder 비교

| 특성 | Encoder-only | Decoder-only | Encoder-Decoder |
|------|-------------|-------------|----------------|
| **대표 모델** | BERT, RoBERTa, ViT | GPT, LLaMA, Claude | T5, BART, Whisper |
| **Attention 방향** | 양방향 (전체 참조) | 단방향 (왼→오) | Encoder: 양방향, Decoder: 단방향+Cross |
| **사전학습** | MLM(마스크 언어 모델), NSP | CLM(다음 토큰 예측) | Span Corruption / Denoising |
| **강점** | 문맥 이해, 분류 | **텍스트 생성** | 입출력이 다른 변환 태스크 |
| **적합한 태스크** | 분류, NER, QA | 대화, 코딩, 요약 | 번역, 요약, TTS |
| **입력 처리** | 전체 문장 한 번에 | 왼쪽→오른쪽 순차 | 입력 전체 → 출력 순차 |
| **출력 방식** | 각 토큰/[CLS] 벡터 | 한 토큰씩 생성 | 한 토큰씩 생성 |
| **현재 트렌드** | NLU 태스크에서 여전히 강함 | **주류** (LLM의 대세) | 특정 태스크에서 활약 |
| **비유** | 독해력 시험의 달인 | 소설 작가 | 동시통역사 |

## 4.3 핵심 수식 요약표

| 수식 | 이름 | 쉬운 설명 |
|------|------|------|
| $Q = XW^Q,\ K = XW^K,\ V = XW^V$ | Q, K, V 생성 | 입력을 검색어(Q), 제목(K), 내용(V)으로 변환 |
| $\text{Attention}(Q,K,V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}})V$ | Scaled Dot-Product Attention | 관련 있는 정보를 찾아서 가중합 |
| $\text{Var}(q \cdot k) = d_k$ | 스케일링 근거 | 내적의 분산이 $d_k$에 비례 → $\sqrt{d_k}$로 나눠 안정화 |
| $\text{MultiHead} = \text{Concat}(\text{head}_1,...,\text{head}_h)W^O$ | Multi-Head Attention | 여러 전문가가 각자 다른 관점으로 분석 |
| $d_k = d_v = d_{model} / h$ | Head별 차원 분할 | 총 연산량은 Single-Head와 동일 |
| $PE(pos,2i) = \sin(pos / 10000^{2i/d_{model}})$ | Positional Encoding | 각 위치에 고유한 지문 부여 |
| $\text{LayerNorm}(x + \text{SubLayer}(x))$ | Add & Norm | 메모 유지(잔차) + 값 정규화 |
| $\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$ | Feed-Forward Network | 본 것을 깊이 처리하는 단계 |
| $\text{softmax}(\frac{QK^T}{\sqrt{d_k}} + M)V$ | Masked Attention | 미래 토큰을 가려서 순차 생성 가능 |

## 4.4 다음 학습 방향

이 노트북에서 배운 내용을 바탕으로:

1. **더 깊이**: 논문 "Attention is All You Need" 원문 읽기
2. **더 넓게**: GPT 계열 (Decoder-only) 구현해보기
3. **실전 활용**: 한국어 BERT (KoBERT, KLUE-BERT)로 한국어 태스크 실습
4. **최신 기법**: LoRA, QLoRA 등 효율적 Fine-tuning(미세조정) 기법 학습
5. **대규모 모델**: LLM (Large Language Model, 거대 언어 모델)의 원리 이해

---

**수고하셨습니다!** Transformer와 BERT의 핵심을 모두 다뤘습니다. 이 내용이 LLM 시대를 이해하는 튼튼한 기초가 되길 바랍니다.

### 이 노트북에서 배운 것 정리

| 배운 것 | 핵심 한 줄 |
|---------|-----------|
| Self-Attention | 모든 단어 쌍의 관계를 한 번에 계산 |
| Multi-Head Attention | 여러 관점에서 동시에 관계 파악 |
| Positional Encoding | 순서 정보가 없는 Transformer에 위치 정보 부여 |
| Encoder Block | Attention + FFN + Residual + LayerNorm |
| Causal Mask | 미래 토큰 차단으로 순차 생성 가능 |
| BERT | 양방향 Encoder로 문맥을 깊이 이해 |
| Fine-tuning | 사전학습 모델을 소량 데이터로 태스크에 적응 |